# Reduced Order Modeling (ROM)

Previously. we approximated the entire chip and cold plate assembly as a single lumped thermal node governed by a single thermal conductance parameter, $UA$. While this approach is computationally inexpensive and highly suitable for real-time control, it assumes infinite thermal conductivity within the solid components. In high-power thermofluidic systems, this Biot analysis fails to capture reality because significant temperature gradients exist across the silicon die, the thermal interface materials (TIM), and the cold plate base.

To accurately predict the maximum internal temperatures without resorting to a computationally expensive 3D Computational Fluid Dynamics (CFD) model, we employ a Reduced Order Model (ROM). A ROM simplifies the spatial complexity of a physical system into a 1D thermal resistance network, analogous to an electrical circuit. In this analogy, the heat dissipated by the chip ($\dot{Q}$) acts as current, temperature differentials ($\Delta T$) act as voltages, and material properties act as thermal resistances ($R$).

For this improved model, we will decompose our system into three distinct solid nodes:
1. **Junction ($T_\text{junc}$)**: The heat-generating silicon die.
2. **Case/Spreader ($T_\text{case}$)**: The top of the chip package.
3. **Cold Plate ($T_\text{cp}$)**: The metallic base absorbing the heat.

The heat path encounters the following thermal resistances:
*   $R_\text{TIM1}$: Resistance between the die and the case.
*   $R_\text{TIM2}$: Resistance between the case and the cold plate.
*   $R_\text{conv}$: Convective resistance between the cold plate and the fluid (equivalent to $1/UA$).

The governing energy balance equations for the steady-state nodal network are:
$$T_\text{case} = T_\text{junc} - \dot{Q} \cdot R_\text{TIM1}$$
$$T_\text{cp} = T_\text{case} - \dot{Q} \cdot R_\text{TIM2}$$
$$T_\text{fluid,avg} = T_\text{cp} - \dot{Q} \cdot R_\text{conv}$$

We will continue to approximate the average fluid temperature as the arithmetic mean of the inlet and outlet, and rely on the sensible heat balance to find the coolant outlet temperature:
$$T_\text{out} = T_\text{in} + \frac{\dot{Q}}{\dot{m} c_p}$$

In [1]:
import numpy as np

class ReducedOrderThermalModel:
    def __init__(self, R_tim1, R_tim2, UA_design, cp=4184):
        """
        1D Thermal Resistance Network Model.
        R_tim1: Thermal resistance of die-to-case TIM (°C/W)
        R_tim2: Thermal resistance of case-to-cold-plate TIM (°C/W)
        UA_design: Design thermal conductance to fluid (W/K)
        cp: Specific heat of the coolant (J/kg-K), default is water
        """
        self.R_tim1 = R_tim1
        self.R_tim2 = R_tim2
        self.R_conv = 1.0 / UA_design
        self.cp = cp
        
    def solve(self, Q, m_dot, T_in):
        """
        Solves for the nodal temperatures across the stack.
        Q: Heat load (W)
        m_dot: Mass flow rate (kg/s)
        T_in: Inlet temperature (°C)
        """
        # Sensible heat balance to find T_out
        T_out = T_in + Q / (m_dot * self.cp)
        
        # Average fluid temperature
        T_fluid_avg = (T_in + T_out) / 2.0
        
        # Calculate temperatures moving upstream from the fluid
        T_cp = T_fluid_avg + (Q * self.R_conv)
        T_case = T_cp + (Q * self.R_tim2)
        T_junc = T_case + (Q * self.R_tim1)
        
        return {
            "T_out": T_out,
            "T_fluid_avg": T_fluid_avg,
            "T_cp": T_cp,
            "T_case": T_case,
            "T_junc": T_junc
        }

# Baseline parameters from the idealized single-node system
Q_load = 1000       # W
m_dot_coolant = 0.05 # kg/s
T_inlet = 20.0      # °C
UA_baseline = 50.0  # W/K

# Define realistic thermal resistances for the internal chip stack
R_tim1_design = 0.005 # °C/W
R_tim2_design = 0.010 # °C/W

# Instantiate and solve the ROM
rom_model = ReducedOrderThermalModel(
    R_tim1=R_tim1_design, 
    R_tim2=R_tim2_design, 
    UA_design=UA_baseline
)

results = rom_model.solve(Q=Q_load, m_dot=m_dot_coolant, T_in=T_inlet)

print("--- Reduced Order Model Predictions ---")
print(f"Coolant Outlet Temp:  {results['T_out']:.2f} °C")
print(f"Cold Plate Base Temp: {results['T_cp']:.2f} °C")
print(f"Chip Case Temp:       {results['T_case']:.2f} °C")
print(f"Chip Junction Temp:   {results['T_junc']:.2f} °C")

--- Reduced Order Model Predictions ---
Coolant Outlet Temp:  24.78 °C
Cold Plate Base Temp: 42.39 °C
Chip Case Temp:       52.39 °C
Chip Junction Temp:   57.39 °C


## Spatiotemporal Reduced Order Modeling 

To capture transient thermal behavior, a data-driven ROM must account for the spatial evolution of the temperature field over time. Instead of sampling only the final steady-state temperatures, this approach samples the transient history to build a spatiotemporal surrogate model.

The high-fidelity full order model (FOM) is run across a parameterized design space of input variables. Each calculated 2D temperature field is flattened into a 1D column vector and concatenated to form a single matrix:
$$\mathbf{S} = [ \mathbf{T}(\dot{Q}_1, t_1), \mathbf{T}(\dot{Q}_1, t_2), \dots, \mathbf{T}(\dot{Q}_m, t_n) ]$$
Where:
*   $\mathbf{S}$: The snapshot matrix containing all recorded temperature states across all sampled parameters and time steps.
*   $\mathbf{T}$: A flattened 1D column vector representing the spatial temperature field at a specific configuration.
*   $\dot{Q}_m$: The $m$-th sampled input heat load (power).
*   $t_n$: The $n$-th discrete time step recorded during the simulation.
*   $m$: The total number of unique power levels simulated.
*   $n$: The total number of discrete time intervals recorded per simulation.

Singular value decomposition (SVD) is applied to the snapshot matrix to extract the system's dominant spatial patterns, known as proper orthogonal decomposition (POD) modes:
$$\mathbf{S} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^T$$
Where:
*   $\mathbf{U}$: A unitary matrix whose columns (left singular vectors) represent the orthonormal spatial modes of the system.
*   $\mathbf{\Sigma}$: A diagonal matrix of singular values, representing the thermal variance (or "energy") captured by each corresponding mode.
*   $\mathbf{V}^T$: The transpose of a unitary matrix whose rows (right singular vectors) represent how the modes evolve with respect to time and the input parameters.

Because heat conduction is highly diffusive, the first few modes capture the vast majority of the system's variance. We truncate the matrices to retain only the most critical modes, discarding the rest to drastically reduce the spatial degrees of freedom.
*   $r$: The number of retained dominant modes (typically $r \le 3$ for linear conduction).
*   $\phi_i(x,y)$: The $i$-th retained spatial mode, corresponding directly to the $i$-th column of $\mathbf{U}$, mapped back to the 2D grid $(x,y)$.

In a transient ROM, the spatial modes remain static, but their corresponding amplitude coefficients must scale dynamically. The transient temperature field is approximated as:
$$T(x,y,t) \approx \sum_{i=1}^{r} a_i(\dot{Q}, t) \phi_i(x,y)$$
Where:
*   $T(x,y,t)$: The approximated spatial temperature field at coordinates $(x,y)$ and time $t$.
*   $a_i(\dot{Q}, t)$: The amplitude coefficient for the $i$-th mode, which acts as a dynamic weighting factor dictated by the input power $\dot{Q}$ and elapsed time $t$.

To function as a predictive surrogate, the ROM must instantly calculate the coefficients for any unseen scenario. First, the original snapshot matrix is projected onto the retained modes to calculate the "true" coefficients for the training data:
$$\mathbf{A} = \mathbf{U}_r^T \mathbf{S}$$
Where:
*   $\mathbf{A}$: The matrix of true mode coefficients for every recorded snapshot.
*   $\mathbf{U}_r^T$: The transpose of the truncated modal matrix, containing only the $r$ retained spatial modes.

Finally, a multidimensional interpolator maps the continuous input space $(\dot{Q}, t)$ directly to the coefficients in $\mathbf{A}$. When a new temperature prediction is requested, the ROM interpolates the coefficients $a_i(\dot{Q}, t)$ and multiplies them by the static spatial modes $\phi_i(x,y)$, reconstructing the full 2D temperature field instantly.

In [3]:
import numpy as np
import time
from scipy.interpolate import LinearNDInterpolator

class Transient2DFOM:
    def __init__(self, nx=30, ny=30, alpha=1e-4, dx=0.01, dy=0.01):
        self.nx, self.ny = nx, ny
        self.alpha = alpha
        self.dx, self.dy = dx, dy
        self.dt = (dx**2 * dy**2) / (2 * alpha * (dx**2 + dy**2)) * 0.8
        
    def solve_and_record(self, Q_input, steps=2000, record_every=50):
        """Solves the 2D heat equation and records snapshots over time."""
        T = np.zeros((self.nx, self.ny))
        T_new = np.zeros((self.nx, self.ny))
        cx, cy = self.nx//2, self.ny//2
        
        snapshots = []
        times = []
        
        start_time = time.perf_counter()
        for step in range(1, steps + 1):
            T[cx-2:cx+3, cy-2:cy+3] += Q_input * self.dt 
            
            T_new[1:-1, 1:-1] = T[1:-1, 1:-1] + self.alpha * self.dt * (
                (T[2:, 1:-1] - 2*T[1:-1, 1:-1] + T[:-2, 1:-1]) / self.dx**2 +
                (T[1:-1, 2:] - 2*T[1:-1, 1:-1] + T[1:-1, :-2]) / self.dy**2
            )
            T = T_new.copy()
            
            if step % record_every == 0 or step == steps:
                snapshots.append(T.flatten())
                times.append(step * self.dt)
                
        end_time = time.perf_counter()
        return np.column_stack(snapshots), np.array(times), end_time - start_time

class SpatiotemporalROM:
    def __init__(self, num_modes=3):
        self.num_modes = num_modes
        self.modes = None
        self.interpolator = None

    def train(self, snapshot_matrix, Q_t_points):
        """Extracts spatial modes and trains an interpolator for (Q, t) -> coefficients."""
        # Perform SVD directly on snapshots (no mean subtraction for zero-reference linear systems)
        U, S, Vh = np.linalg.svd(snapshot_matrix, full_matrices=False)
        self.modes = U[:, :self.num_modes]
        
        # Project all snapshots onto the retained modes to get the true coefficients
        # Shape: (num_modes, num_snapshots)
        coefficients = self.modes.T @ snapshot_matrix
        
        # Train a multidimensional interpolator: Inputs (Q, t) -> Outputs (Mode Coefficients)
        self.interpolator = LinearNDInterpolator(Q_t_points, coefficients.T)

    def predict(self, Q_input, t_input):
        """Predicts the temperature field instantaneously for a given Q and time."""
        start_time = time.perf_counter()
        
        # Predict mode coefficients for the requested (Q, t)
        predicted_coeffs = self.interpolator(Q_input, t_input)
        
        if np.isnan(predicted_coeffs).any():
            raise ValueError("Requested (Q, t) is outside the training hull.")
            
        # Reconstruct the full 2D field
        T_pred = self.modes @ predicted_coeffs
        
        end_time = time.perf_counter()
        return T_pred, end_time - start_time

# --- 1. Generate Extensive Transient Training Data ---
print("Generating comprehensive FOM snapshots for training...")
fom = Transient2DFOM(nx=30, ny=30)
training_Qs = [10.0, 50.0, 100.0, 150.0]
total_steps = 2000
record_interval = 40

all_snapshots = []
Q_t_points = []

for Q in training_Qs:
    snaps, times, _ = fom.solve_and_record(Q, steps=total_steps, record_every=record_interval)
    all_snapshots.append(snaps)
    for t in times:
        Q_t_points.append([Q, t])

snapshot_matrix = np.hstack(all_snapshots)
Q_t_points = np.array(Q_t_points)

print(f"Collected {snapshot_matrix.shape[1]} transient snapshots.")

# --- 2. Train the Transient ROM ---
print("Extracting modes and training Spatiotemporal ROM...")
rom = SpatiotemporalROM(num_modes=3)
rom.train(snapshot_matrix, Q_t_points)

# --- 3. Compare FOM vs ROM on an Unseen Test Scenario ---
test_Q = 85.0
print(f"\n--- Testing Scenario: Q = {test_Q} W at final timestep ---")

# Run FOM to get the ground truth at the final step
test_snaps, test_times, fom_time = fom.solve_and_record(test_Q, steps=total_steps, record_every=total_steps)
fom_final_field = test_snaps[:, -1]
target_time = test_times[-1]

# Run ROM to predict the exact same point in time
rom_final_field, rom_time = rom.predict(test_Q, target_time)

# Metrics
max_temp_fom = np.max(fom_final_field)
max_temp_rom = np.max(rom_final_field)
error = np.linalg.norm(fom_final_field - rom_final_field) / np.linalg.norm(fom_final_field) * 100
speedup = fom_time / rom_time

print(f"FOM Execution Time: {fom_time:.5f} seconds | Max Temp: {max_temp_fom:.2f} °C")
print(f"ROM Execution Time: {rom_time:.5f} seconds | Max Temp: {max_temp_rom:.2f} °C")
print(f"Prediction Error (L2 Norm): {error:.4f}%")
print(f"Computational Speedup: {speedup:.1f}x")

Generating comprehensive FOM snapshots for training...
Collected 200 transient snapshots.
Extracting modes and training Spatiotemporal ROM...

--- Testing Scenario: Q = 85.0 W at final timestep ---
FOM Execution Time: 0.02204 seconds | Max Temp: 735.02 °C
ROM Execution Time: 0.00083 seconds | Max Temp: 735.12 °C
Prediction Error (L2 Norm): 0.0281%
Computational Speedup: 26.5x


## Appendix A

The full order model script solves the 2D transient heat conduction equation to simulate how thermal energy diffuses through a solid medium over time when subjected to a central heat source. The physical system is governed by the following partial differential equation:

$$\frac{\partial T}{\partial t} = \alpha \left( \frac{\partial^2 T}{\partial x^2} + \frac{\partial^2 T}{\partial y^2} \right) + g'''$$

where $T$ is the temperature field, $\alpha$ is the thermal diffusivity of the material, and $g'''$ represents the discrete heat generation term applied to the center of the grid. To solve this, the continuous domain is discretized using the forward-time central-space (FTCS) finite difference method. This is an explicit numerical scheme that steps forward in time. The temperature of an interior node at the next time step ($n+1$) is calculated using the current spatial temperature gradients between itself and its four immediate neighbors:
$$T_{i,j}^{n+1} = T_{i,j}^n + \alpha \Delta t \left( \frac{T_{i+1,j}^n - 2T_{i,j}^n + T_{i-1,j}^n}{\Delta x^2} + \frac{T_{i,j+1}^n - 2T_{i,j}^n + T_{i,j-1}^n}{\Delta y^2} \right)$$
The time step $\Delta t$ is calculated automatically  to strictly satisfy the stability criterion for a 2D explicit solver, preventing numerical divergence.

The grid is initialized to a uniform reference temperature of $T(x,y,0) = 0$. The model employs constant Dirichlet boundary conditions on all four edges. The perimeter of the spatial domain is pinned to $T=0$, acting as a constant-temperature heat sink that absorbs the thermal energy diffusing outward from the center.

## Apendix B

Assume we test 5 heating levels and record 6 time steps for each, you generate $5 \times 6 = 30$ total snapshots. Because you only have 4 spatial nodes, the maximum rank (the maximum number of independent spatial patterns) of this system is 4. When generating ROMs, algorithms use the economy-size SVD to discard the empty mathematical null space, keeping the matrices manageable. Here is the exact visual breakdown of the SVD equation $\mathbf{S} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^T$ for your specific setup.

The Snapshot Matrix ($\mathbf{S}$). Dimensions: $4 \times 30$ (4 rows for spatial nodes, 30 columns for each recorded state).
This is your raw training data. Each column is the flattened temperature field of the 4 nodes at a specific power ($\dot{Q}$) and time ($t$).

$$\mathbf{S} = \begin{bmatrix} T_{1}(\dot{Q}_1, t_1) & T_{1}(\dot{Q}_1, t_2) & \dots & T_{1}(\dot{Q}_5, t_6) \\ T_{2}(\dot{Q}_1, t_1) & T_{2}(\dot{Q}_1, t_2) & \dots & T_{2}(\dot{Q}_5, t_6) \\ T_{3}(\dot{Q}_1, t_1) & T_{3}(\dot{Q}_1, t_2) & \dots & T_{3}(\dot{Q}_5, t_6) \\ T_{4}(\dot{Q}_1, t_1) & T_{4}(\dot{Q}_1, t_2) & \dots & T_{4}(\dot{Q}_5, t_6) \end{bmatrix}$$

The Unitary Matrix of Spatial Modes ($\mathbf{U}$). Dimensions: $4 \times 4$. The columns of this matrix are your fundamental spatial modes ($\phi_1, \phi_2, \phi_3, \phi_4$). Because there are only 4 spatial nodes in the physical grid, there can only be 4 spatial blueprints. They are completely independent of time and power.

$$\mathbf{U} = \begin{bmatrix} \vert{} & \vert{} & \vert{} & \vert{} \\ \phi_1 & \phi_2 & \phi_3 & \phi_4 \\ \vert{} & \vert{} & \vert{} & \vert{} \end{bmatrix} = \begin{bmatrix} u_{1,1} & u_{1,2} & u_{1,3} & u_{1,4} \\ u_{2,1} & u_{2,2} & u_{2,3} & u_{2,4} \\ u_{3,1} & u_{3,2} & u_{3,3} & u_{3,4} \\ u_{4,1} & u_{4,2} & u_{4,3} & u_{4,4} \end{bmatrix}$$

The Diagonal Matrix of Singular Values ($\mathbf{\Sigma}$). Dimensions: $4 \times 4$. This matrix dictates the hierarchical importance of the spatial modes in $\mathbf{U}$. The values are sorted from largest ($\sigma_1$) to smallest ($\sigma_4$). If $\sigma_4$ is close to zero, it means the shape $\phi_4$ barely contributes to the physics.

$$\mathbf{\Sigma} = \begin{bmatrix} \sigma_1 & 0 & 0 & 0 \\ 0 & \sigma_2 & 0 & 0 \\ 0 & 0 & \sigma_3 & 0 \\ 0 & 0 & 0 & \sigma_4 \end{bmatrix}$$

The Transposed Unitary Matrix of Coefficients ($\mathbf{V}^T$). Dimensions: $4 \times 30$. This matrix maps the history. Each row corresponds to one of the spatial modes in $\mathbf{U}$. The 30 columns correspond to your 30 original snapshots. For example, the first row tells you exactly how much the first spatial mode ($\phi_1$) was "turned on" during Snapshot 1, Snapshot 2, etc.

$$\mathbf{V}^T = \begin{bmatrix} v_{1,1} & v_{1,2} & \dots & v_{1,30} \\ v_{2,1} & v_{2,2} & \dots & v_{2,30} \\ v_{3,1} & v_{3,2} & \dots & v_{3,30} \\ v_{4,1} & v_{4,2} & \dots & v_{4,30} \end{bmatrix}$$

*(Note: The true mode coefficients matrix $\mathbf{A}$ used for training the interpolator is calculated by multiplying $\mathbf{\Sigma} \mathbf{V}^T$.)*

Applying Truncation ($r$): If you decide to retain only the first two dominant modes ($r=2$) to build your ROM, you physically slice the matrices. You discard the right half of $\mathbf{U}$ and $\mathbf{\Sigma}$, and the bottom half of $\mathbf{V}^T$:

* $\mathbf{U}_r$ becomes a $4 \times 2$ matrix (keeping only $\phi_1$ and $\phi_2$).
* $\mathbf{\Sigma}_r$ becomes a $2 \times 2$ matrix (keeping only $\sigma_1$ and $\sigma_2$).
* $\mathbf{V}_r^T$ becomes a $2 \times 30$ matrix.

By truncating, you sacrifice a tiny fraction of accuracy (discarding the negligible shapes $\phi_3$ and $\phi_4$) to cut the computational load in half.

Note that we are not solving for these variables algebraically by guessing or balancing an equation. SVD is a deterministic linear algebra algorithm. When you feed your snapshot matrix $\mathbf{S}$ into an SVD solver (like `numpy.linalg.svd` in Python), the algorithm calculates $\mathbf{U}$, $\mathbf{\Sigma}$, and $\mathbf{V}^T$ entirely from the internal data correlations within $\mathbf{S}$. It does this by turning your snapshot matrix into a covariance problem. 

Determining $\phi$: To find the fundamental spatial shapes, the algorithm calculates the matrix product $\mathbf{S} \mathbf{S}^T$. Because $\mathbf{S}$ is $4 \times 30$, its transpose $\mathbf{S}^T$ is $30 \times 4$. Multiplying them together yields a $4 \times 4$ matrix. Physically, this is the spatial covariance matrix. It compares every spatial node against every other spatial node across all 30 snapshots to see how their temperatures move together. The algorithm then calculates the eigenvectors of this $4 \times 4$ covariance matrix. These resulting eigenvectors are the columns of $\mathbf{U}$. They are your $\phi_1, \phi_2, \phi_3, \phi_4$ spatial modes.

Determining $\sigma$ (The Singular Values in $\mathbf{\Sigma}$): When the algorithm calculates the eigenvectors of $\mathbf{S} \mathbf{S}^T$, it also calculates their corresponding eigenvalues. An eigenvalue represents the total thermal variance (or "energy") captured by its corresponding eigenvector. To get the singular values ($\sigma_1, \sigma_2$, etc.) that populate the diagonal of $\mathbf{\Sigma}$, the algorithm simply takes the square root of these eigenvalues. They are then sorted from largest to smallest, which forces the most important spatial modes to the front of $\mathbf{U}$.

Determining $v$ (The Coefficients in $\mathbf{V}^T$): To find how these spatial modes evolve across the 30 snapshots, the algorithm calculates the matrix product in reverse: $\mathbf{S}^T \mathbf{S}$. This yields a $30 \times 30$ matrix. Physically, this is the temporal covariance matrix. Instead of comparing spatial nodes, it compares every snapshot against every other snapshot to find similarities in time and power. The algorithm calculates the eigenvectors of this $30 \times 30$ matrix. These eigenvectors form the columns of $\mathbf{V}$ (which become the rows of $\mathbf{V}^T$). The values $v_{i,j}$ simply represent how strongly the $i$-th spatial mode is activated during the $j$-th snapshot. By breaking $\mathbf{S}$ down into these self-referential covariance matrices, the SVD algorithm guarantees that multiplying $\mathbf{U} \mathbf{\Sigma} \mathbf{V}^T$ will perfectly reconstruct your original $\mathbf{S}$ without you ever having to manually define a single variable.